# Tráfico, contaminación y electrificación en Madrid (2019–2025)
### Análisis integrado de calidad del aire, movilidad y transición vehicular

**Asignatura:** Visualización y Analítica de Datos — UNED  
**Herramienta:** Altair (Python 3)  
**Fuentes:** datos.madrid.es · datos.gob.es (DGT)

---

## ▶ Cómo ejecutar este notebook

1. Haz clic en **Entorno de ejecución → Ejecutar todo** (o `Ctrl+F9`)
2. Cuando aparezca el diálogo de Google Drive, haz clic en **Conectar con Google Drive** y autoriza el acceso
3. La primera ejecución descarga y procesa los datos (~10 min). Las siguientes cargan directamente desde Drive.

> **Para el evaluador:** si ya se ha ejecutado antes y los datos están en Drive, la celda de descarga los detecta automáticamente y los omite. El dashboard interactivo aparece al final del notebook.


## 1. Instalación de dependencias

In [ ]:
# Las siguientes librerías no vienen preinstaladas en Colab
%pip install -q altair==5.3.0 vl-convert-python geopandas tqdm

import importlib, sys
for lib in ['altair', 'geopandas', 'tqdm']:
    v = importlib.import_module(lib).__version__
    print(f"{lib}: {v}")
print("✓ Dependencias listas")


## 2. Conexión con Google Drive

Los datos se guardan en tu Google Drive en la carpeta:  
`Mi unidad / madrid_trafico_aire_ev / data/`

Así la descarga solo ocurre una vez. Las siguientes ejecuciones cargan desde Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Carpeta raíz del proyecto en Drive
DRIVE_ROOT = Path('/content/drive/MyDrive/madrid_trafico_aire_ev')
RAW_DIR    = DRIVE_ROOT / 'data' / 'raw'
PROC_DIR   = DRIVE_ROOT / 'data' / 'processed'

for d in [RAW_DIR/'aire', RAW_DIR/'meteo', RAW_DIR/'trafico',
          RAW_DIR/'dgt', PROC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"✓ Google Drive montado")
print(f"  Datos en: {DRIVE_ROOT}")


## 3. Imports y configuración global

In [ ]:
import os, sys, time, zipfile, io, warnings
import hashlib, logging
from datetime import datetime

import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm.notebook import tqdm
import altair as alt

warnings.filterwarnings('ignore')
alt.data_transformers.enable('vegafusion')  # necesario para datasets grandes en Colab

# ── Magnitudes de calidad del aire (código → nombre) ──────────────────────────
MAGNITUDES = {
    1:  'SO2',   6:  'CO',   7:  'NO',   8:  'NO2',
    9:  'PM25',  10: 'PM10', 12: 'NOx',  14: 'O3',
    20: 'TOL',   22: 'BEN',  30: 'SO2_2',42: 'CO_2',
    44: 'HC',    431:'PM25_2'
}

# ── Parámetros meteorológicos ─────────────────────────────────────────────────
METEO_MAG = {
    81: 'vel_viento',  82: 'dir_viento',  83: 'temperatura',
    86: 'humedad',     87: 'presion',      88: 'rad_solar',
    89: 'precipitacion'
}

AÑOS = list(range(2019, 2026))
print("✓ Configuración lista")
print(f"  Período de análisis: {AÑOS[0]}–{AÑOS[-1]}")


## 4. Descarga de datos

Cada fuente se descarga solo si no existe ya en Drive. Si la descarga ya se realizó, esta sección se ejecuta en segundos.


In [ ]:
TIMEOUT  = 120
REINTENTOS = 3
HEADERS  = {'User-Agent': 'Mozilla/5.0 (academic research UNED)'}

def descargar(url, destino, desc=""):
    """Descarga url→destino. Si ya existe y no está vacío, lo omite."""
    destino = Path(destino)
    if destino.exists() and destino.stat().st_size > 1000:
        print(f"  ✓ Ya existe: {destino.name} ({destino.stat().st_size/1e6:.1f} MB)")
        return True
    destino.parent.mkdir(parents=True, exist_ok=True)
    for intento in range(1, REINTENTOS + 1):
        try:
            r = requests.get(url, stream=True, timeout=TIMEOUT, headers=HEADERS)
            if r.status_code == 404:
                print(f"  ✗ 404: {url}")
                return False
            r.raise_for_status()
            total = int(r.headers.get('Content-Length', 0))
            with open(destino, 'wb') as f, tqdm(
                total=total, unit='B', unit_scale=True,
                desc=f"  {desc or destino.name}", leave=False
            ) as bar:
                for chunk in r.iter_content(65536):
                    f.write(chunk); bar.update(len(chunk))
            print(f"  ✓ {destino.name}  ({destino.stat().st_size/1e6:.1f} MB)")
            return True
        except Exception as e:
            print(f"  ⚠ Intento {intento}/{REINTENTOS}: {e}")
            if destino.exists(): destino.unlink()
            if intento < REINTENTOS: time.sleep(5 * intento)
    return False

def unzip(src, out_dir):
    try:
        with zipfile.ZipFile(src) as z:
            z.extractall(out_dir)
    except zipfile.BadZipFile:
        print(f"  ⚠ ZIP corrupto: {src}")

# ── URLs de descarga verificadas (31/03/2026) ─────────────────────────────────
AIRE_URLS = {
    2019: ("https://datos.madrid.es/egob/catalogo/201200-7-calidad-aire-horario.zip",  "zip"),
    2020: ("https://datos.madrid.es/egob/catalogo/201200-8-calidad-aire-horario.zip",  "zip"),
    2021: ("https://datos.madrid.es/egob/catalogo/201200-9-calidad-aire-horario.zip",  "zip"),
    2022: ("https://datos.madrid.es/egob/catalogo/201200-10-calidad-aire-horario.zip", "zip"),
    2023: ("https://datos.madrid.es/egob/catalogo/201200-11-calidad-aire-horario.zip", "zip"),
    2024: ("https://datos.madrid.es/egob/catalogo/201200-12-calidad-aire-horario.zip", "zip"),
    2025: ("https://datos.madrid.es/egob/catalogo/201200-1-calidad-aire-horario-csv.csv", "csv"),
}
METEO_URLS = {
    2019: ("https://datos.madrid.es/egob/catalogo/300392-7-meteorologia-meteoro.zip",  "zip"),
    2020: ("https://datos.madrid.es/egob/catalogo/300392-8-meteorologia-meteoro.zip",  "zip"),
    2021: ("https://datos.madrid.es/egob/catalogo/300392-9-meteorologia-meteoro.zip",  "zip"),
    2022: ("https://datos.madrid.es/egob/catalogo/300392-10-meteorologia-meteoro.zip", "zip"),
    2023: ("https://datos.madrid.es/egob/catalogo/300392-11-meteorologia-meteoro.zip", "zip"),
    2024: ("https://datos.madrid.es/egob/catalogo/300392-12-meteorologia-meteoro.zip", "zip"),
    2025: ("https://datos.madrid.es/egob/catalogo/300392-1-meteorologia-meteoro-csv.csv", "csv"),
}

print("=== F1 — Calidad del Aire ===")
for año, (url, ext) in AIRE_URLS.items():
    dest = RAW_DIR / 'aire' / f'aire_{año}.{ext}'
    if descargar(url, dest, f"Aire {año}") and ext == 'zip':
        unzip(dest, RAW_DIR / 'aire' / str(año))
    time.sleep(0.5)

print("\n=== F2 — Meteorología ===")
for año, (url, ext) in METEO_URLS.items():
    dest = RAW_DIR / 'meteo' / f'meteo_{año}.{ext}'
    if descargar(url, dest, f"Meteo {año}") and ext == 'zip':
        unzip(dest, RAW_DIR / 'meteo' / str(año))
    time.sleep(0.5)

# Estaciones auxiliares
descargar(
    "https://datos.madrid.es/egob/catalogo/212629-2-estaciones-control-aire.csv",
    RAW_DIR / 'aire' / 'estaciones_aire.csv', "Estaciones aire"
)
descargar(
    "https://datos.madrid.es/egob/catalogo/202468-0-puntos-medida-trafico.csv",
    RAW_DIR / 'trafico' / 'puntos_medida.csv', "Puntos medida tráfico"
)
print("\n✓ F1 y F2 completadas")


In [ ]:
print("=== F3 — Tráfico histórico (84 ficheros ZIP, puede tardar 20-30 min) ===")
print("Nota: los ZIPs de tráfico son ~85 MB/mes. Si Drive ya los tiene, se omiten.\n")

errores_trafico = []
for año in AÑOS:
    for mes in range(1, 13):
        url  = (f"https://datos.madrid.es/egob/catalogo/"
                f"208627-{año}{mes:02d}-trafico-historico.zip")
        dest = RAW_DIR / 'trafico' / str(año) / f"trafico_{año}{mes:02d}.zip"
        ok = descargar(url, dest, f"Tráfico {año}-{mes:02d}")
        if ok:
            unzip(dest, RAW_DIR / 'trafico' / str(año))
        else:
            errores_trafico.append(f"{año}-{mes:02d}")
        time.sleep(0.8)

if errores_trafico:
    print(f"\n⚠ {len(errores_trafico)} ficheros con error: {errores_trafico}")
else:
    print("\n✓ F3 Tráfico completada sin errores")


In [ ]:
print("=== F4 — Matriculaciones DGT (MATRABA) ===")
print("Nota: cada fichero anual es ~500 MB–1 GB. Descarga lenta la primera vez.\n")

for año in AÑOS:
    url  = (f"https://sedeapl.dgt.gob.es/IEST_INTER/pdfs/estadistica/"
            f"vehiculos/matriculaciones/{año}/MATRABA_{año}.txt")
    dest = RAW_DIR / 'dgt' / f"MATRABA_{año}.txt"
    descargar(url, dest, f"DGT {año}")
    time.sleep(1.0)

print("\n✓ F4 DGT completada")


## 5. Procesado y limpieza de datos

In [ ]:
# ── 5.1  Calidad del Aire: wide → long, filtrar 2019-2025 ──────────────────────

PROC_AIRE = PROC_DIR / 'df_aire_diario.parquet'

if PROC_AIRE.exists():
    print(f"✓ Cargando desde caché: {PROC_AIRE}")
    df_aire = pd.read_parquet(PROC_AIRE)
else:
    dfs = []
    for año in AÑOS:
        # Buscar CSV dentro del ZIP descomprimido o el CSV directo
        carpeta = RAW_DIR / 'aire' / str(año)
        csvs = list(carpeta.glob('*.csv')) if carpeta.exists() else []
        csv_directo = RAW_DIR / 'aire' / f'aire_{año}.csv'
        if csv_directo.exists(): csvs.append(csv_directo)
        for csv in csvs:
            try:
                df = pd.read_csv(csv, sep=';', encoding='latin-1', low_memory=False)
                df['año_src'] = año
                dfs.append(df)
            except Exception as e:
                print(f"  ⚠ Error leyendo {csv}: {e}")

    if not dfs:
        print("⚠ No se encontraron ficheros de aire. Ejecuta primero la celda de descarga.")
    else:
        raw = pd.concat(dfs, ignore_index=True)
        print(f"  Filas brutas: {len(raw):,}")

        # Detectar columnas de hora (H01…H24) y columnas id
        hora_cols = [c for c in raw.columns if c.upper().startswith('H') and
                     c[1:].isdigit() and 1 <= int(c[1:]) <= 24]
        id_cols   = ['ESTACION', 'MAGNITUD', 'PUNTO_MUESTREO',
                     'ANO', 'MES', 'DIA']
        id_cols   = [c for c in id_cols if c in raw.columns]

        df_long = raw.melt(id_vars=id_cols, value_vars=hora_cols,
                           var_name='hora_str', value_name='valor')
        df_long['hora'] = df_long['hora_str'].str.extract(r'(\d+)').astype(int) - 1
        df_long['datetime'] = pd.to_datetime(dict(
            year=df_long['ANO'], month=df_long['MES'],
            day=df_long['DIA'], hour=df_long['hora']
        ), errors='coerce')

        # Filtrar solo magnitudes de interés
        df_long = df_long[df_long['MAGNITUD'].isin(MAGNITUDES.keys())].copy()
        df_long['contaminante'] = df_long['MAGNITUD'].map(MAGNITUDES)

        # Valores inválidos → NaN
        df_long['valor'] = pd.to_numeric(df_long['valor'], errors='coerce')
        df_long.loc[df_long['valor'] < 0, 'valor'] = np.nan
        df_long.loc[df_long['valor'] > 1000, 'valor'] = np.nan  # límite físico

        # Agregar a diario por estación + contaminante
        df_aire = (df_long.groupby(['ESTACION', 'contaminante',
                                    df_long['datetime'].dt.date.rename('fecha')])
                          ['valor']
                          .agg(['mean', 'max', 'count'])
                          .rename(columns={'mean':'media','max':'maximo','count':'n_horas'})
                          .reset_index())
        df_aire['fecha'] = pd.to_datetime(df_aire['fecha'])

        # Solo NO2, PM10, PM25, O3
        df_aire = df_aire[df_aire['contaminante'].isin(['NO2','PM10','PM25','O3'])]
        df_aire.to_parquet(PROC_AIRE, index=False)
        print(f"  ✓ df_aire guardado: {len(df_aire):,} filas → {PROC_AIRE}")

df_aire.head(3)


In [ ]:
# ── 5.2  Meteorología: wide → long → diario ────────────────────────────────────

PROC_METEO = PROC_DIR / 'df_meteo_diario.parquet'

if PROC_METEO.exists():
    print(f"✓ Cargando desde caché: {PROC_METEO}")
    df_meteo = pd.read_parquet(PROC_METEO)
else:
    dfs = []
    for año in AÑOS:
        carpeta = RAW_DIR / 'meteo' / str(año)
        csvs = list(carpeta.glob('*.csv')) if carpeta.exists() else []
        csv_directo = RAW_DIR / 'meteo' / f'meteo_{año}.csv'
        if csv_directo.exists(): csvs.append(csv_directo)
        for csv in csvs:
            try:
                df = pd.read_csv(csv, sep=';', encoding='latin-1', low_memory=False)
                dfs.append(df)
            except Exception as e:
                print(f"  ⚠ {csv}: {e}")

    if dfs:
        raw = pd.concat(dfs, ignore_index=True)
        hora_cols = [c for c in raw.columns if c.upper().startswith('H') and
                     c[1:].isdigit() and 1 <= int(c[1:]) <= 24]
        id_cols = [c for c in ['ESTACION','MAGNITUD','ANO','MES','DIA'] if c in raw.columns]

        df_long = raw.melt(id_vars=id_cols, value_vars=hora_cols,
                           var_name='hora_str', value_name='valor')
        df_long = df_long[df_long['MAGNITUD'].isin(METEO_MAG.keys())].copy()
        df_long['variable'] = df_long['MAGNITUD'].map(METEO_MAG)
        df_long['valor']    = pd.to_numeric(df_long['valor'], errors='coerce')
        df_long['hora']     = df_long['hora_str'].str.extract(r'(\d+)').astype(int) - 1
        df_long['datetime'] = pd.to_datetime(dict(
            year=df_long['ANO'], month=df_long['MES'],
            day=df_long['DIA'], hour=df_long['hora']
        ), errors='coerce')

        df_meteo = (df_long.groupby(['variable', df_long['datetime'].dt.date.rename('fecha')])
                           ['valor'].agg(['mean','max','min'])
                           .rename(columns={'mean':'media','max':'maximo','min':'minimo'})
                           .reset_index())
        df_meteo['fecha'] = pd.to_datetime(df_meteo['fecha'])
        df_meteo.to_parquet(PROC_METEO, index=False)
        print(f"  ✓ df_meteo guardado: {len(df_meteo):,} filas → {PROC_METEO}")

df_meteo.head(3)


In [ ]:
# ── 5.3  Tráfico: 15-min → horario → diario ───────────────────────────────────

PROC_TRAFICO = PROC_DIR / 'df_trafico_diario.parquet'

if PROC_TRAFICO.exists():
    print(f"✓ Cargando desde caché: {PROC_TRAFICO}")
    df_trafico = pd.read_parquet(PROC_TRAFICO)
else:
    dfs = []
    for año in AÑOS:
        carpeta = RAW_DIR / 'trafico' / str(año)
        if not carpeta.exists(): continue
        for csv in carpeta.glob('*.csv'):
            try:
                df = pd.read_csv(csv, sep=';', encoding='latin-1',
                                 low_memory=False,
                                 usecols=lambda c: c in
                                     ['idelem','fecha','hora','intensidad','ocupacion','carga'])
                dfs.append(df)
            except Exception as e:
                print(f"  ⚠ {csv.name}: {e}")
        print(f"  Año {año}: {len(dfs)} ficheros acumulados")

    if dfs:
        raw = pd.concat(dfs, ignore_index=True)
        raw.columns = raw.columns.str.lower()
        raw['datetime'] = pd.to_datetime(
            raw['fecha'].astype(str) + ' ' + raw['hora'].astype(str),
            format='%Y%m%d %H:%M', errors='coerce'
        )
        # Agregar a horario
        raw_h = (raw.groupby(['idelem', raw['datetime'].dt.floor('H').rename('datetime_h')])
                    [['intensidad','ocupacion']]
                    .mean().reset_index())
        # Agregar a diario
        df_trafico = (raw_h.groupby(['idelem', raw_h['datetime_h'].dt.date.rename('fecha')])
                           [['intensidad','ocupacion']]
                           .mean().reset_index())
        df_trafico['fecha'] = pd.to_datetime(df_trafico['fecha'])
        df_trafico.to_parquet(PROC_TRAFICO, index=False)
        print(f"\n  ✓ df_trafico guardado: {len(df_trafico):,} filas → {PROC_TRAFICO}")

df_trafico.head(3)


In [ ]:
# ── 5.4  DGT Matriculaciones: filtrar Madrid, clasificar propulsión ────────────

PROC_DGT = PROC_DIR / 'df_matriculaciones_mensual.parquet'

if PROC_DGT.exists():
    print(f"✓ Cargando desde caché: {PROC_DGT}")
    df_mat = pd.read_parquet(PROC_DGT)
else:
    dfs = []
    for año in AÑOS:
        f = RAW_DIR / 'dgt' / f'MATRABA_{año}.txt'
        if not f.exists(): continue
        try:
            df = pd.read_csv(f, sep=';', encoding='latin-1', low_memory=False,
                             on_bad_lines='skip')
            # Filtrar provincia de Madrid (código 28)
            prov_col = [c for c in df.columns if 'PROVINCIA' in c.upper() and 'MATR' in c.upper()]
            if prov_col:
                df = df[df[prov_col[0]].astype(str).str.startswith('28')]
            # Seleccionar columnas útiles
            keep = [c for c in df.columns if any(k in c.upper() for k in
                    ['PROPUL','FECHA_MAT','TIPO_VEH','CATEG'])]
            df = df[keep].copy()
            df['año'] = año
            dfs.append(df)
            print(f"  ✓ DGT {año}: {len(df):,} registros (Madrid)")
        except Exception as e:
            print(f"  ⚠ DGT {año}: {e}")

    if dfs:
        raw = pd.concat(dfs, ignore_index=True)
        # Normalizar columna de propulsión
        prop_col = [c for c in raw.columns if 'PROPUL' in c.upper()][0]
        raw['propulsion_raw'] = raw[prop_col].astype(str).str.upper().str.strip()

        def clasificar(p):
            if any(x in p for x in ['ELECTR','BEV','EL ','ELE']): return 'BEV'
            if any(x in p for x in ['HIBR','PLUG','PHEV','HE']):   return 'PHEV'
            if 'HIBRID' in p:                                        return 'Hibrido_noenchufable'
            if 'GASOL' in p:                                         return 'Gasolina'
            if 'DIESE' in p or 'GASOIL' in p:                       return 'Diesel'
            return 'Otro'

        raw['tipo_propulsion'] = raw['propulsion_raw'].apply(clasificar)

        # Extraer mes de matriculación
        fecha_col = [c for c in raw.columns if 'FECHA' in c.upper() and 'MAT' in c.upper()][0]
        raw['fecha_mat'] = pd.to_datetime(raw[fecha_col].astype(str), errors='coerce', format='%Y%m')
        raw = raw.dropna(subset=['fecha_mat'])
        raw['mes'] = raw['fecha_mat'].dt.to_period('M').astype(str)

        df_mat = (raw.groupby(['mes','tipo_propulsion'])
                     .size().reset_index(name='n_matriculaciones'))
        df_mat['fecha'] = pd.to_datetime(df_mat['mes'])
        df_mat.to_parquet(PROC_DGT, index=False)
        print(f"\n  ✓ df_mat guardado: {len(df_mat):,} filas → {PROC_DGT}")

df_mat.head(6)


## 6. Dashboard interactivo (Altair)

7 visualizaciones interactivas. Haz clic, arrastra y pasa el ratón por encima para explorar los datos.


In [ ]:
# ── Configuración visual ────────────────────────────────────────────────────────
SCHEME   = 'tableau10'
W, H     = 700, 300
W_WIDE   = 950

alt.themes.enable('default')

# ── Preparar datos para visualizaciones ───────────────────────────────────────

# NO2 diario por estación (tipo de estación del fichero auxiliar)
no2 = df_aire[df_aire['contaminante'] == 'NO2'].copy()
no2['año'] = no2['fecha'].dt.year
no2['mes'] = no2['fecha'].dt.to_period('M').astype(str)
no2['dia_semana'] = no2['fecha'].dt.day_name()
no2['hora_dummy'] = 12  # para heatmap

# Media diaria de tráfico (todos los sensores)
trafico_d = df_trafico.groupby('fecha')['intensidad'].mean().reset_index()
trafico_d.columns = ['fecha', 'intensidad_media']

# Meteo diaria (temperatura y viento)
temp_d  = df_meteo[df_meteo['variable'] == 'temperatura'][['fecha','media']].rename(columns={'media':'temp'})
viento_d = df_meteo[df_meteo['variable'] == 'vel_viento'][['fecha','media']].rename(columns={'media':'viento'})

# Matriculaciones EV + PHEV vs. total
electrico = df_mat[df_mat['tipo_propulsion'].isin(['BEV','PHEV'])].copy()
total_mat  = df_mat.groupby('fecha')['n_matriculaciones'].sum().reset_index(name='total')
ev_mes     = (electrico.groupby('fecha')['n_matriculaciones'].sum()
                       .reset_index(name='ev_phev'))
ev_cuota   = ev_mes.merge(total_mat, on='fecha')
ev_cuota['pct_ev'] = 100 * ev_cuota['ev_phev'] / ev_cuota['total']

print("✓ Datos preparados para visualización")


In [ ]:
# ── VIZ 1: Evolución anual de NO2 ─────────────────────────────────────────────
no2_anual = (no2.groupby(['año', 'ESTACION'])['media']
               .mean().reset_index()
               .rename(columns={'media':'no2_medio'}))

selector_est = alt.selection_point(fields=['ESTACION'], bind='legend')

v1 = alt.Chart(no2_anual, title='① Evolución anual de NO₂ por estación').mark_line(
    point=True, strokeWidth=2
).encode(
    x=alt.X('año:O', title='Año'),
    y=alt.Y('no2_medio:Q', title='NO₂ medio anual (µg/m³)'),
    color=alt.Color('ESTACION:N', title='Estación', scale=alt.Scale(scheme=SCHEME)),
    opacity=alt.condition(selector_est, alt.value(1), alt.value(0.15)),
    tooltip=['ESTACION:N', 'año:O', alt.Tooltip('no2_medio:Q', format='.1f')]
).add_params(selector_est).properties(width=W, height=H)

v1


In [ ]:
# ── VIZ 2: Heatmap hora × día de semana (NO2 medio) ──────────────────────────
no2_hora = no2.copy()
no2_hora['hora'] = no2_hora['fecha'].dt.hour  # hora de la fecha
no2_hora['dow']  = no2_hora['fecha'].dt.dayofweek

ORDER_DOW = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
ORDER_DOW_ES = ['Lun','Mar','Mié','Jue','Vie','Sáb','Dom']

no2_heat = (no2.assign(
    dow=no2['fecha'].dt.day_name(),
    mes_num=no2['fecha'].dt.month
).groupby(['dow','mes_num'])['media'].mean().reset_index()
 .rename(columns={'media':'no2_medio'}))

v2 = alt.Chart(no2_heat, title='② Heatmap NO₂: mes × día de semana').mark_rect().encode(
    x=alt.X('mes_num:O', title='Mes', axis=alt.Axis(labelExpr=
        "['','Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic'][datum.value]")),
    y=alt.Y('dow:N', title='Día', sort=ORDER_DOW),
    color=alt.Color('no2_medio:Q', title='NO₂ (µg/m³)',
                    scale=alt.Scale(scheme='orangered')),
    tooltip=['dow:N', 'mes_num:O', alt.Tooltip('no2_medio:Q', format='.1f')]
).properties(width=W, height=H)

v2


In [ ]:
# ── VIZ 3: Tráfico vs. NO2 (scatter diario con brushing) ─────────────────────
no2_total = (no2.groupby('fecha')['media'].mean()
                .reset_index().rename(columns={'media':'no2_medio'}))
merged = (no2_total.merge(trafico_d, on='fecha')
                   .merge(temp_d, on='fecha', how='left'))
merged['año'] = merged['fecha'].dt.year.astype(str)
merged = merged.dropna(subset=['no2_medio','intensidad_media'])

brush = alt.selection_interval()

v3_scatter = alt.Chart(
    merged.sample(min(5000, len(merged)), random_state=42),
    title='③ Tráfico vs. NO₂ (muestra diaria)'
).mark_circle(opacity=0.4, size=25).encode(
    x=alt.X('intensidad_media:Q', title='Intensidad tráfico media (veh/h)'),
    y=alt.Y('no2_medio:Q', title='NO₂ medio diario (µg/m³)'),
    color=alt.condition(brush,
        alt.Color('año:N', scale=alt.Scale(scheme=SCHEME)),
        alt.value('lightgray')),
    tooltip=['fecha:T', alt.Tooltip('no2_medio:Q', format='.1f'),
             alt.Tooltip('intensidad_media:Q', format='.0f'), 'año:N']
).add_params(brush).properties(width=W, height=H)

v3_hist = alt.Chart(merged.sample(min(5000, len(merged)), random_state=42)).mark_bar().encode(
    x=alt.X('no2_medio:Q', bin=alt.Bin(maxbins=30), title='NO₂ (µg/m³)'),
    y=alt.Y('count():Q', title='Días'),
    color=alt.condition(brush, alt.value('#2E75B6'), alt.value('lightgray'))
).transform_filter(brush).properties(width=W, height=100)

v3 = alt.vconcat(v3_scatter, v3_hist)
v3


In [ ]:
# ── VIZ 4: Efecto COVID-19 ────────────────────────────────────────────────────
# Comparar tráfico y NO2 en 2019, 2020, 2021
covid = merged[merged['año'].isin(['2019','2020','2021'])].copy()
covid['dia_año'] = covid['fecha'].dt.dayofyear

base = alt.Chart(covid)

v4_no2 = base.mark_line(strokeWidth=1.5, opacity=0.8).encode(
    x=alt.X('dia_año:Q', title='Día del año'),
    y=alt.Y('no2_medio:Q', title='NO₂ (µg/m³)'),
    color=alt.Color('año:N', scale=alt.Scale(scheme='set1')),
    tooltip=['año:N', 'dia_año:Q', alt.Tooltip('no2_medio:Q', format='.1f')]
).properties(width=W, height=180, title='④ Impacto COVID-19 en NO₂')

# Anotación confinamiento
confinamiento = alt.Chart(pd.DataFrame({
    'x': [75], 'x2': [172], 'label': ['Confinamiento
(15 mar–21 jun 2020)']
})).mark_rect(opacity=0.15, color='red').encode(
    x='x:Q', x2='x2:Q'
) + alt.Chart(pd.DataFrame({'x':[75], 'y':[60], 'label':['← Confinamiento 2020']}
)).mark_text(align='left', color='red', fontSize=10).encode(x='x:Q', y='y:Q', text='label:N')

v4_traf = base.mark_line(strokeWidth=1.5, opacity=0.8).encode(
    x=alt.X('dia_año:Q', title='Día del año'),
    y=alt.Y('intensidad_media:Q', title='Intensidad tráfico (veh/h)'),
    color=alt.Color('año:N', scale=alt.Scale(scheme='set1'))
).properties(width=W, height=180, title='④ Impacto COVID-19 en tráfico')

v4 = alt.vconcat(v4_no2 + confinamiento, v4_traf + confinamiento)
v4


In [ ]:
# ── VIZ 5: Scatter temperatura vs. NO2 (color = viento) ──────────────────────
meteo_merged = (merged.merge(viento_d, on='fecha', how='left'))
muestra = meteo_merged.dropna(subset=['temp','viento','no2_medio']).sample(
    min(4000, len(meteo_merged)), random_state=7)

v5 = alt.Chart(muestra, title='⑤ Temperatura vs. NO₂ (color = velocidad del viento)').mark_circle(
    size=30, opacity=0.5
).encode(
    x=alt.X('temp:Q', title='Temperatura (ºC)'),
    y=alt.Y('no2_medio:Q', title='NO₂ (µg/m³)'),
    color=alt.Color('viento:Q', title='Viento (km/h)',
                    scale=alt.Scale(scheme='blues', reverse=True)),
    tooltip=[alt.Tooltip('temp:Q',format='.1f'),
             alt.Tooltip('no2_medio:Q',format='.1f'),
             alt.Tooltip('viento:Q',format='.1f'), 'fecha:T']
).properties(width=W, height=H)

v5


In [ ]:
# ── VIZ 6: Evolución cuota EV+PHEV en Madrid ─────────────────────────────────
v6a = alt.Chart(ev_cuota, title='⑥ Crecimiento EV+PHEV en Madrid y cuota sobre total').mark_area(
    color='#2E75B6', opacity=0.7
).encode(
    x=alt.X('fecha:T', title=''),
    y=alt.Y('ev_phev:Q', title='Matriculaciones EV+PHEV/mes'),
    tooltip=['fecha:T', 'ev_phev:Q', alt.Tooltip('pct_ev:Q', format='.1f', title='% cuota')]
).properties(width=W, height=160)

v6b = alt.Chart(ev_cuota).mark_line(color='orange', strokeWidth=2).encode(
    x=alt.X('fecha:T', title=''),
    y=alt.Y('pct_ev:Q', title='% cuota electrificada'),
    tooltip=['fecha:T', alt.Tooltip('pct_ev:Q', format='.2f')]
).properties(width=W, height=100)

v6 = alt.vconcat(v6a, v6b)
v6


In [ ]:
# ── VIZ 7: Tendencia EV vs. NO2 medio anual ───────────────────────────────────
no2_anual_global = (no2.groupby(no2['fecha'].dt.year.rename('año'))['media']
                       .mean().reset_index().rename(columns={'media':'no2_medio'}))
ev_anual = (ev_cuota.assign(año=ev_cuota['fecha'].dt.year)
                    .groupby('año')['pct_ev'].mean().reset_index())
tendencia = no2_anual_global.merge(ev_anual, on='año').dropna()

v7 = alt.Chart(tendencia, title='⑦ Cuota EV anual vs. NO₂ medio — tendencia 2019-2025').mark_point(
    size=120, filled=True
).encode(
    x=alt.X('pct_ev:Q', title='% cuota electrificada anual (Madrid)'),
    y=alt.Y('no2_medio:Q', title='NO₂ medio anual (µg/m³)'),
    color=alt.Color('año:O', scale=alt.Scale(scheme='viridis'), title='Año'),
    tooltip=['año:O',
             alt.Tooltip('pct_ev:Q', format='.2f', title='% EV'),
             alt.Tooltip('no2_medio:Q', format='.1f', title='NO₂ µg/m³')]
).properties(width=500, height=H) + alt.Chart(tendencia).mark_line(
    color='gray', strokeDash=[4,2]
).encode(
    x='pct_ev:Q', y='no2_medio:Q'
).transform_regression('pct_ev', 'no2_medio')

v7


In [ ]:
# ── Dashboard completo combinado ─────────────────────────────────────────────
print("Generando dashboard completo...")

dashboard = alt.vconcat(
    alt.hconcat(v1, v2).resolve_scale(color='independent'),
    alt.hconcat(v3_scatter, v4_no2).resolve_scale(color='independent'),
    alt.hconcat(v5, v6a).resolve_scale(color='independent'),
    v7
).properties(
    title=alt.TitleParams(
        text='Tráfico, contaminación y electrificación en Madrid (2019–2025)',
        subtitle='Análisis integrado — Altair — UNED',
        fontSize=18, subtitleFontSize=13
    )
).configure_view(stroke=None)

# Guardar HTML en Drive (para entrega)
out_html = DRIVE_ROOT / 'dashboard_madrid.html'
dashboard.save(str(out_html))
print(f"✓ Dashboard guardado en Drive: {out_html}")
print(f"  Comparte ese fichero para que el evaluador pueda abrirlo directamente.")

dashboard


## 7. Entrega

El evaluador puede acceder a este proyecto de tres formas, **sin ejecutar nada en local**:

| Opción | URL | Requisito |
|---|---|---|
| **Abrir en Colab** (ejecución completa) | Enlace al notebook en GitHub | Cuenta Google |
| **Ver dashboard HTML** | Enlace público en Drive | Ninguno |
| **Ver notebook con outputs** | nbviewer.org | Ninguno |

**Para publicar el enlace de Colab:**  
1. Sube este `.ipynb` a un repositorio GitHub público  
2. Abre `https://colab.research.google.com/github/{usuario}/{repo}/blob/main/Madrid_Trafico_Contaminacion_EV.ipynb`  
3. Ese enlace es el que incluyes en la memoria

**Para el dashboard HTML estático:**  
1. Ve a Google Drive → `madrid_trafico_aire_ev/dashboard_madrid.html`  
2. Clic derecho → Compartir → Cualquier persona con el enlace puede ver  
3. Ese enlace abre el dashboard interactivo sin necesidad de ejecutar nada
